## **NER in Python: Pre-Trained & Custom Models BY NeuralLine**

src: https://www.youtube.com/watch?v=JIz-hiRrZ2g

In [1]:
import spacy

In [16]:
texts = [
  'John goes for a walk in Berlin',
  'Mike is going to the store',
  'Elon Musk is the CEO at twitter',
  'Bob Smith is the guy behing XYZ-Soft Inc.',
  'Florian Dedov is the guy behing NeuralNine'
]

In [5]:
import en_core_web_sm
nlp = en_core_web_sm.load()

In [9]:
ner_labels = nlp.get_pipe('ner').labels

In [12]:
ner_labels

('CARDINAL',
 'DATE',
 'EVENT',
 'FAC',
 'GPE',
 'LANGUAGE',
 'LAW',
 'LOC',
 'MONEY',
 'NORP',
 'ORDINAL',
 'ORG',
 'PERCENT',
 'PERSON',
 'PRODUCT',
 'QUANTITY',
 'TIME',
 'WORK_OF_ART')

In [14]:
categories = ['ORG', 'PERSON', 'LOC']

In [17]:
docs = [nlp(text) for text in texts]

In [20]:
for doc in docs:
  entities = []
  for ent in doc.ents:
    # if ent.label_ in categories:
      entities.append((ent.text, ent.label_))
  print(entities)

[('John', 'PERSON'), ('Berlin', 'GPE')]
[('Mike', 'PERSON')]
[('Elon Musk', 'PERSON')]
[('Bob Smith', 'PERSON'), ('XYZ-Soft Inc.', 'ORG')]
[('Florian', 'NORP'), ('NeuralNine', 'ORG')]


In [22]:
products = [
  'What is the price of 4 bananas',
  'How much are 16 chairs',
  'Give me the value of 5 laptops',
  'Give me the value of five laptops'
]

In [ ]:
# import en_core_web_sm
# nlp = en_core_web_sm.load()

In [23]:
docs = [nlp(text) for text in products]

In [24]:
for doc in docs:
  entities = []
  for ent in doc.ents:
    # if ent.label_ in categories:
      entities.append((ent.text, ent.label_))
  print(entities)

[('4', 'CARDINAL')]
[('16', 'CARDINAL')]
[('5', 'CARDINAL')]
[('five', 'CARDINAL')]


## **Fine tuning Spacy**

In [19]:
import random

import spacy
from spacy.util import minibatch
from spacy.training import Example

In [20]:
train_data = [
    ("What is the price of 10 bananas?", {"entities": [(21, 23, "QUANTITY"), (24, 31, "PRODUCT")]}),
    ("I need 5 apples and 2 oranges.", {"entities": [(7, 8, "QUANTITY"), (9, 15, "PRODUCT"), (20, 21, "QUANTITY"), (22, 29, "PRODUCT")]}),
    ("How much for 3 kilograms of rice?", {"entities": [(10, 11, "QUANTITY"), (12, 23, "UNIT"), (27, 31, "PRODUCT")]}),
    ("Can I get 2 loaves of bread?", {"entities": [(9, 10, "QUANTITY"), (11, 17, "UNIT"), (21, 26, "PRODUCT")]}),
    ("Give me 4 liters of milk.", {"entities": [(8, 9, "QUANTITY"), (10, 16, "UNIT"), (20, 24, "PRODUCT")]}),
    ("I want 6 packets of chips.", {"entities": [(7, 8, "QUANTITY"), (9, 16, "UNIT"), (20, 25, "PRODUCT")]}),
    ("Buy 12 eggs for me.", {"entities": [(4, 6, "QUANTITY"), (7, 11, "PRODUCT")]}),
    ("Add 2 kilograms of sugar to the cart.", {"entities": [(4, 5, "QUANTITY"), (6, 17, "UNIT"), (21, 26, "PRODUCT")]}),
    ("Purchase 3 cans of soda.", {"entities": [(9, 10, "QUANTITY"), (11, 15, "UNIT"), (19, 23, "PRODUCT")]}),
    ("How much does 1 dozen donuts cost?", {"entities": [(14, 15, "QUANTITY"), (16, 21, "UNIT"), (22, 28, "PRODUCT")]}),
    ("We need 10 bottles of water.", {"entities": [(8, 10, "QUANTITY"), (11, 18, "UNIT"), (22, 27, "PRODUCT")]}),
    ("Bring 7 bars of chocolate.", {"entities": [(6, 7, "QUANTITY"), (8, 12, "UNIT"), (16, 25, "PRODUCT")]}),
    ("Order 15 kilograms of potatoes.", {"entities": [(6, 8, "QUANTITY"), (9, 20, "UNIT"), (24, 32, "PRODUCT")]}),
    ("Please buy 9 apples and 4 oranges.", {"entities": [(11, 12, "QUANTITY"), (13, 19, "PRODUCT"), (24, 25, "QUANTITY"), (26, 33, "PRODUCT")]}),
    ("Get 20 liters of juice.", {"entities": [(4, 6, "QUANTITY"), (7, 13, "UNIT"), (17, 22, "PRODUCT")]}),
    ("Pick up 3 kilograms of tomatoes.", {"entities": [(8, 9, "QUANTITY"), (10, 21, "UNIT"), (25, 33, "PRODUCT")]}),
    ("I need 8 rolls of tissue paper.", {"entities": [(7, 8, "QUANTITY"), (9, 14, "UNIT"), (18, 30, "PRODUCT")]}),
    ("Bring me 4 bags of flour.", {"entities": [(9, 10, "QUANTITY"), (11, 15, "UNIT"), (19, 24, "PRODUCT")]}),
    ("I would like 10 packets of biscuits.", {"entities": [(14, 16, "QUANTITY"), (17, 24, "UNIT"), (28, 36, "PRODUCT")]}),
    ("Get 3 bottles of olive oil.", {"entities": [(4, 5, "QUANTITY"), (6, 13, "UNIT"), (17, 26, "PRODUCT")]}),
    ("Buy 5 kilograms of chicken.", {"entities": [(4, 5, "QUANTITY"), (6, 17, "UNIT"), (21, 28, "PRODUCT")]}),
    ("Purchase 6 cartons of milk.", {"entities": [(9, 10, "QUANTITY"), (11, 18, "UNIT"), (22, 26, "PRODUCT")]}),
    ("I need 2 packs of pasta.", {"entities": [(7, 8, "QUANTITY"), (9, 14, "UNIT"), (18, 23, "PRODUCT")]}),
    ("Please bring 7 liters of oil.", {"entities": [(13, 14, "QUANTITY"), (15, 21, "UNIT"), (25, 28, "PRODUCT")]}),
    ("Pick up 9 cans of beans.", {"entities": [(8, 9, "QUANTITY"), (10, 14, "UNIT"), (18, 23, "PRODUCT")]}),
    ("Order 4 loaves of bread.", {"entities": [(6, 7, "QUANTITY"), (8, 14, "UNIT"), (18, 23, "PRODUCT")]}),
    ("Buy 3 dozen eggs for breakfast.", {"entities": [(4, 5, "QUANTITY"), (6, 11, "UNIT"), (12, 16, "PRODUCT")]}),
    ("How much for 5 kilograms of onions?", {"entities": [(10, 11, "QUANTITY"), (12, 23, "UNIT"), (27, 33, "PRODUCT")]}),
    ("I need 4 packets of salt.", {"entities": [(7, 8, "QUANTITY"), (9, 16, "UNIT"), (20, 24, "PRODUCT")]}),
    ("Add 2 cans of soup to the list.", {"entities": [(4, 5, "QUANTITY"), (6, 10, "UNIT"), (14, 18, "PRODUCT")]}),
    ("Bring 6 boxes of cereal.", {"entities": [(6, 7, "QUANTITY"), (8, 13, "UNIT"), (17, 23, "PRODUCT")]}),
    ("Order 7 liters of orange juice.", {"entities": [(6, 7, "QUANTITY"), (8, 14, "UNIT"), (18, 30, "PRODUCT")]}),
    ("Buy 8 bars of soap.", {"entities": [(4, 5, "QUANTITY"), (6, 10, "UNIT"), (14, 18, "PRODUCT")]}),
    ("Get 12 packets of coffee.", {"entities": [(4, 6, "QUANTITY"), (7, 14, "UNIT"), (18, 24, "PRODUCT")]}),
    ("How much for 9 kilograms of sugar?", {"entities": [(10, 11, "QUANTITY"), (12, 23, "UNIT"), (27, 32, "PRODUCT")]}),
    ("Please buy 3 jars of honey.", {"entities": [(11, 12, "QUANTITY"), (13, 17, "UNIT"), (21, 26, "PRODUCT")]}),
    ("I need 11 packets of tea.", {"entities": [(7, 9, "QUANTITY"), (10, 17, "UNIT"), (21, 24, "PRODUCT")]}),
    ("Bring 14 cans of soda.", {"entities": [(6, 8, "QUANTITY"), (9, 13, "UNIT"), (17, 21, "PRODUCT")]}),
    ("Get me 5 kilograms of mangoes.", {"entities": [(7, 8, "QUANTITY"), (9, 20, "UNIT"), (24, 31, "PRODUCT")]}),
    ("Purchase 8 cartons of eggs.", {"entities": [(9, 10, "QUANTITY"), (11, 18, "UNIT"), (22, 26, "PRODUCT")]}),
    ("Can I get 6 boxes of tissues?", {"entities": [(9, 10, "QUANTITY"), (11, 16, "UNIT"), (20, 27, "PRODUCT")]}),
    ("Add 3 bottles of ketchup.", {"entities": [(4, 5, "QUANTITY"), (6, 13, "UNIT"), (17, 24, "PRODUCT")]}),
    ("Buy 7 packs of cookies.", {"entities": [(4, 5, "QUANTITY"), (6, 11, "UNIT"), (15, 22, "PRODUCT")]}),
    ("How much for 10 cans of tuna?", {"entities": [(10, 12, "QUANTITY"), (13, 17, "UNIT"), (21, 25, "PRODUCT")]}),
    ("Bring me 5 liters of vinegar.", {"entities": [(9, 10, "QUANTITY"), (11, 17, "UNIT"), (21, 28, "PRODUCT")]}),
    ("Order 6 packets of sugar.", {"entities": [(6, 7, "QUANTITY"), (8, 15, "UNIT"), (19, 24, "PRODUCT")]}),
    ("Buy 2 loaves of rye bread.", {"entities": [(4, 5, "QUANTITY"), (6, 12, "UNIT"), (16, 25, "PRODUCT")]}),
]


In [21]:
# nlp = spacy.load("en_core_web_md")
import en_core_web_sm
nlp = en_core_web_sm.load()

In [27]:
import spacy  # Import the spaCy library
from spacy.training import Example  # Import the Example class for creating training examples
from spacy.util import minibatch  # Import the minibatch utility for batching data
import random  # Import the random module for shuffling data

# Load or create pipeline
nlp = spacy.blank("en")  # Create a blank pipeline for the English language

# Check if the NER (Named Entity Recognition) component is in the pipeline
if 'ner' not in nlp.pipe_names:
    ner = nlp.add_pipe('ner', last=True)  # Add the NER component to the pipeline if it doesn't exist
else:
    ner = nlp.get_pipe('ner')  # Retrieve the existing NER component

# Add entity labels from the training data to the NER component
for _, annotations in train_data:  # Iterate through the training data
    for ent in annotations.get("entities"):  # Access the list of entities in the annotations
        ner.add_label(ent[2])  # Add the entity label (ent[2]) to the NER component

# Disable other components in the pipeline during training to focus on NER
other_pipes = [pipe for pipe in nlp.pipe_names if pipe != 'ner']  # Get the names of all components except NER
with nlp.disable_pipes(*other_pipes):  # Temporarily disable other pipeline components
    nlp.initialize()  # Initialize the pipeline (required for spaCy 3.x)

    epochs = 50  # Number of training epochs (iterations over the dataset)

    # Training loop
    for epoch in range(epochs):  # Iterate through each epoch
        random.shuffle(train_data)  # Shuffle the training data to avoid overfitting
        losses = {}  # Dictionary to store the training loss for each epoch

        # Create batches from the training data
        batches = minibatch(train_data, size=2)  # Generate batches of size 2
        for batch in batches:  # Iterate through each batch
            examples = []  # List to store Example objects for the current batch
            for text, annotations in batch:  # Iterate through the text and annotations in the batch
                doc = nlp.make_doc(text)  # Create a spaCy Doc object from the text
                examples.append(Example.from_dict(doc, annotations))  # Create an Example object and append it

            # Update the NER component with the current batch of examples
            nlp.update(
                examples,  # Pass the list of Example objects
                drop=0.5,  # Dropout rate (regularization to prevent overfitting)
                losses=losses  # Track the loss for the current batch
            )
        # Print the loss for the current epoch
        print(f'Epoch: {epoch + 1}, Loss: {losses}')

Epoch: 1, Loss: {'ner': 201.37266323715448}
Epoch: 2, Loss: {'ner': 150.96957743404164}
Epoch: 3, Loss: {'ner': 96.27906588556664}
Epoch: 4, Loss: {'ner': 46.24271531459692}
Epoch: 5, Loss: {'ner': 34.818076391771626}
Epoch: 6, Loss: {'ner': 25.836204546356203}
Epoch: 7, Loss: {'ner': 23.99814733781015}
Epoch: 8, Loss: {'ner': 23.581229968416338}
Epoch: 9, Loss: {'ner': 16.83110092519015}
Epoch: 10, Loss: {'ner': 20.380297293499183}
Epoch: 11, Loss: {'ner': 20.59757337407223}
Epoch: 12, Loss: {'ner': 12.126716291999003}
Epoch: 13, Loss: {'ner': 17.85140113570023}
Epoch: 14, Loss: {'ner': 9.963617890655808}
Epoch: 15, Loss: {'ner': 5.461035614052955}
Epoch: 16, Loss: {'ner': 6.900959570918258}
Epoch: 17, Loss: {'ner': 4.597149835873471}
Epoch: 18, Loss: {'ner': 8.583456260732676}
Epoch: 19, Loss: {'ner': 9.843724593356736}
Epoch: 20, Loss: {'ner': 8.470587365763755}
Epoch: 21, Loss: {'ner': 2.879514581144624}
Epoch: 22, Loss: {'ner': 2.1261377773308814}
Epoch: 23, Loss: {'ner': 0.646057

In [28]:
nlp.to_disk('custom_ner_model')

In [29]:
trained_nlp = spacy.load('custom_ner_model')

In [30]:
test_texts = [
    "How much for 3 oranges?",
    "I need 15 chairs for the conference.",
    "Can you give the me the price for 6 desks?"
]

In [31]:
for text in test_texts:
  doc = trained_nlp(text)
  print(f'Text: {text}')
  print(f'Entities: {[(ent.text, ent.label_) for ent in doc.ents]}')

Text: How much for 3 oranges?
Entities: [('3', 'QUANTITY'), ('oranges', 'PRODUCT')]
Text: I need 15 chairs for the conference.
Entities: [('15', 'QUANTITY'), ('chairs', 'PRODUCT')]
Text: Can you give the me the price for 6 desks?
Entities: [('6', 'QUANTITY'), ('desks', 'PRODUCT')]
